# Real-World Data Preprocessing

This notebook demonstrates how to preprocess real-world data before applying LAGO.

**Common preprocessing steps:**
1. Map node labels (names, IDs) to integer node IDs
2. Transform date strings to Unix timestamps
3. Normalize timestamps to consecutive integers
4. Apply LAGO
5. Map results back to original labels

**Use case:** Email exchanges, meeting logs, social media interactions, etc.

In [ ]:
import pandas as pd
from datetime import datetime
from lago import LinkStream, lago_modules
from lago.viz import LongitudinalModulesPlot
import matplotlib.pyplot as plt

%matplotlib inline

## 1. Raw Data with String Labels and Dates

Simulate real-world interaction data (e.g., email exchanges).

In [ ]:
# Raw interaction data: (sender, receiver, timestamp)
raw_data = [
    # Project Alpha team (early period)
    ("Alice", "Bob", "2024-01-15 09:30:00"),
    ("Bob", "Carol", "2024-01-15 10:15:00"),
    ("Alice", "Carol", "2024-01-15 14:20:00"),
    ("Alice", "Bob", "2024-01-16 11:00:00"),
    ("Carol", "Bob", "2024-01-16 15:30:00"),
    
    # Project Beta team (early period)
    ("Dave", "Eve", "2024-01-15 08:45:00"),
    ("Eve", "Frank", "2024-01-15 13:00:00"),
    ("Dave", "Frank", "2024-01-16 09:20:00"),
    ("Eve", "Frank", "2024-01-16 16:00:00"),
    
    # Cross-team collaboration (mid period)
    ("Carol", "Dave", "2024-01-17 10:30:00"),
    ("Alice", "Eve", "2024-01-17 14:00:00"),
    
    # Continued interactions (later period)
    ("Alice", "Bob", "2024-01-18 09:00:00"),
    ("Dave", "Eve", "2024-01-18 11:30:00"),
    ("Bob", "Carol", "2024-01-18 15:00:00"),
    ("Eve", "Frank", "2024-01-19 10:00:00"),
    ("Alice", "Carol", "2024-01-19 13:30:00"),
]

# Convert to DataFrame for easier manipulation
df = pd.DataFrame(raw_data, columns=["source", "target", "timestamp"])
print("Raw interaction data:")
print(df.head(10))
print(f"\nTotal interactions: {len(df)}")
print(f"Unique people: {sorted(set(df['source']) | set(df['target']))}")

## 2. Step 1: Map Node Labels to Integer IDs

LAGO requires integer node IDs. Create a bidirectional mapping.

In [ ]:
# Get all unique node labels
unique_nodes = sorted(set(df['source']) | set(df['target']))

# Create mappings: label <-> ID
label_to_id = {label: idx for idx, label in enumerate(unique_nodes)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print("Node mappings:")
for label, node_id in label_to_id.items():
    print(f"  {label:8s} -> {node_id}")

# Map node labels to IDs
df['source_id'] = df['source'].map(label_to_id)
df['target_id'] = df['target'].map(label_to_id)

print("\nData with mapped IDs:")
print(df[['source', 'source_id', 'target', 'target_id', 'timestamp']].head())

## 3. Step 2: Transform Date Strings to Unix Timestamps

Convert date strings to numeric timestamps.

In [ ]:
# Parse date strings and convert to Unix timestamps
df['timestamp_dt'] = pd.to_datetime(df['timestamp'])
df['timestamp_unix'] = df['timestamp_dt'].astype(int) // 10**9  # Convert to seconds

print("Date to Unix timestamp conversion:")
print(df[['timestamp', 'timestamp_dt', 'timestamp_unix']].head())
print(f"\nTime range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Unix range: {df['timestamp_unix'].min()} to {df['timestamp_unix'].max()}")

## 4. Step 3: Normalize Timestamps to Consecutive Integers

LAGO works best with small integer timestamps. Normalize to start from 0.

In [ ]:
# Sort by timestamp
df = df.sort_values('timestamp_unix').reset_index(drop=True)

# Get unique timestamps and create mapping
unique_timestamps = sorted(df['timestamp_unix'].unique())
timestamp_to_int = {ts: idx for idx, ts in enumerate(unique_timestamps)}
int_to_timestamp = {idx: ts for ts, idx in timestamp_to_int.items()}

# Map to normalized integers
df['time_normalized'] = df['timestamp_unix'].map(timestamp_to_int)

print("Timestamp normalization:")
print(df[['timestamp', 'timestamp_unix', 'time_normalized']].head(10))
print(f"\nNormalized time range: {df['time_normalized'].min()} to {df['time_normalized'].max()}")
print(f"Number of unique time points: {len(unique_timestamps)}")

## 5. Create LinkStream from Preprocessed Data

Now the data is ready for LAGO!

In [ ]:
# Extract preprocessed edges: (source_id, target_id, time_normalized)
edges = df[['source_id', 'target_id', 'time_normalized']].values.tolist()

# Create LinkStream
ls = LinkStream()
ls.add_links(edges)

print(f"LinkStream created:")
print(f"  Nodes: {ls.nb_nodes}")
print(f"  Time-edges: {ls.nb_time_edges}")
print(f"  Time steps: {ls.nb_timesteps}")
print(f"  Nodes: {sorted(ls.nodes)}")

## 6. Detect Communities

Apply LAGO on the preprocessed data.

In [ ]:
# Detect temporal communities
communities = lago_modules(ls, omega=2, gamma=1)

print(f"Detected {communities.nb_modules} communities")
print("\nCommunity details (with integer IDs):")
for module in communities.iter_modules():
    print(f"  Community {module.label}:")
    print(f"    Nodes (IDs): {sorted(module.nodes)}")
    print(f"    Duration: {module.duration} time steps")

## 7. Map Results Back to Original Labels

Convert integer IDs back to meaningful names.

In [ ]:
print("Community details (with original labels):\n")
for module in communities.iter_modules():
    # Map node IDs back to labels
    node_labels = [id_to_label[node_id] for node_id in sorted(module.nodes)]
    
    # Map time points back to dates
    time_points = sorted(module.times)
    if time_points:
        start_unix = int_to_timestamp[time_points[0]]
        end_unix = int_to_timestamp[time_points[-1]]
        start_date = datetime.fromtimestamp(start_unix).strftime('%Y-%m-%d')
        end_date = datetime.fromtimestamp(end_unix).strftime('%Y-%m-%d')
    else:
        start_date = end_date = "N/A"
    
    print(f"Community {module.label}:")
    print(f"  Members: {', '.join(node_labels)}")
    print(f"  Period: {start_date} to {end_date}")
    print(f"  Duration: {module.duration} time steps")
    print()

## 8. Track Individual Trajectories

See how specific people move between communities.

In [ ]:
# Track a specific person (e.g., "Alice")
person_name = "Alice"
person_id = label_to_id[person_name]

print(f"Tracking {person_name} (ID: {person_id}):\n")

memberships = communities.get_modules_of_node(person_id)
for m in memberships:
    time_points = sorted(m.times)
    dates = [datetime.fromtimestamp(int_to_timestamp[t]).strftime('%Y-%m-%d') 
             for t in time_points]
    print(f"  Community {m.module}: {dates[0]} to {dates[-1]}")

# Track all people
print("\n" + "="*50)
print("All trajectories:\n")
for label, node_id in sorted(label_to_id.items()):
    trajectory = communities.get_node_trajectory(node_id)
    print(f"  {label:8s}: {trajectory}")

## 9. Visualization with Original Labels

Create a plot using the original names.

In [ ]:
# Create plot with original labels
plot = LongitudinalModulesPlot(communities, linkstream=ls, width=1400, height=600)

# Configure with original names
plot.configure_nodes(
    labels=id_to_label,  # Map IDs back to names
    auto_ordering=True,
    fontsize=11,
)

plot.configure_edges(
    show_activity=True,
    activity_alpha=0.4,
)

plot.configure_modules(
    color_palette="Set2",
    height=0.8,
)

plot.configure_display(
    padding_bottom=0.5,
    padding_top=0.5,
    show_xlabel=True,
    show_ylabel=True,
)

fig, ax = plot.draw(return_ax=True)
ax.set_title("Email Interaction Communities", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## 10. Reusable Preprocessing Functions

Wrap the preprocessing steps into reusable functions.

In [ ]:
def preprocess_temporal_network(interactions, source_col='source', target_col='target', 
                                time_col='timestamp', date_format=None):
    """
    Preprocess temporal network data with string labels and date timestamps.
    
    Parameters
    ----------
    interactions : list of tuples or DataFrame
        Raw interaction data: (source, target, timestamp)
    source_col : str
        Column name for source nodes
    target_col : str
        Column name for target nodes
    time_col : str
        Column name for timestamps
    date_format : str, optional
        Date format string (if None, pandas will infer)
    
    Returns
    -------
    linkstream : LinkStream
        Preprocessed linkstream ready for LAGO
    label_to_id : dict
        Mapping from original labels to integer IDs
    id_to_label : dict
        Mapping from integer IDs to original labels
    time_to_int : dict
        Mapping from Unix timestamps to normalized integers
    int_to_time : dict
        Mapping from normalized integers to Unix timestamps
    """
    # Convert to DataFrame if needed
    if not isinstance(interactions, pd.DataFrame):
        df = pd.DataFrame(interactions, columns=[source_col, target_col, time_col])
    else:
        df = interactions.copy()
    
    # 1. Map node labels to IDs
    unique_nodes = sorted(set(df[source_col]) | set(df[target_col]))
    label_to_id = {label: idx for idx, label in enumerate(unique_nodes)}
    id_to_label = {idx: label for label, idx in label_to_id.items()}
    
    df['source_id'] = df[source_col].map(label_to_id)
    df['target_id'] = df[target_col].map(label_to_id)
    
    # 2. Convert dates to Unix timestamps
    df['timestamp_dt'] = pd.to_datetime(df[time_col], format=date_format)
    df['timestamp_unix'] = df['timestamp_dt'].astype(int) // 10**9
    
    # 3. Normalize timestamps to consecutive integers
    df = df.sort_values('timestamp_unix').reset_index(drop=True)
    unique_timestamps = sorted(df['timestamp_unix'].unique())
    time_to_int = {ts: idx for idx, ts in enumerate(unique_timestamps)}
    int_to_time = {idx: ts for ts, idx in time_to_int.items()}
    
    df['time_normalized'] = df['timestamp_unix'].map(time_to_int)
    
    # 4. Create LinkStream
    edges = df[['source_id', 'target_id', 'time_normalized']].values.tolist()
    ls = LinkStream()
    ls.add_links(edges)
    
    return ls, label_to_id, id_to_label, time_to_int, int_to_time


def communities_to_original_labels(communities, id_to_label, int_to_time):
    """
    Convert community results back to original labels.
    
    Parameters
    ----------
    communities : TimeModules
        Detected communities
    id_to_label : dict
        Mapping from integer IDs to original labels
    int_to_time : dict
        Mapping from normalized integers to Unix timestamps
    
    Returns
    -------
    results : list of dict
        Community information with original labels
    """
    results = []
    
    for module in communities.iter_modules():
        # Map node IDs to labels
        members = [id_to_label[node_id] for node_id in sorted(module.nodes)]
        
        # Map time points to dates
        time_points = sorted(module.times)
        if time_points:
            start_unix = int_to_time[time_points[0]]
            end_unix = int_to_time[time_points[-1]]
            start_date = datetime.fromtimestamp(start_unix)
            end_date = datetime.fromtimestamp(end_unix)
        else:
            start_date = end_date = None
        
        results.append({
            'community_id': module.label,
            'members': members,
            'start_date': start_date,
            'end_date': end_date,
            'duration': module.duration,
            'cohesion': module.cohesion,
        })
    
    return results


print("Preprocessing functions defined!")
print("  - preprocess_temporal_network()")
print("  - communities_to_original_labels()")

## 11. Example: Using the Preprocessing Functions

Complete pipeline in just a few lines.

In [ ]:
# Start with raw data
raw_interactions = [
    ("Alice", "Bob", "2024-01-15 09:30:00"),
    ("Bob", "Carol", "2024-01-15 10:15:00"),
    ("Dave", "Eve", "2024-01-15 08:45:00"),
    # ... more interactions
]

# Preprocess in one step
ls, label_to_id, id_to_label, time_to_int, int_to_time = preprocess_temporal_network(
    raw_data,  # Using our earlier raw_data
    source_col='source',
    target_col='target',
    time_col='timestamp'
)

print(f"Preprocessed LinkStream: {ls.nb_nodes} nodes, {ls.nb_time_edges} time-edges")

# Detect communities
communities = lago_modules(ls)

# Convert results back
results = communities_to_original_labels(communities, id_to_label, int_to_time)

print(f"\nDetected {len(results)} communities:")
for r in results:
    print(f"\nCommunity {r['community_id']}:")
    print(f"  Members: {', '.join(r['members'])}")
    print(f"  Period: {r['start_date'].strftime('%Y-%m-%d')} to {r['end_date'].strftime('%Y-%m-%d')}")
    print(f"  Duration: {r['duration']} time steps")

---

## Summary

This notebook showed the complete preprocessing pipeline:

1. **Node mapping:** String labels → Integer IDs
2. **Time conversion:** Date strings → Unix timestamps → Normalized integers
3. **LAGO application:** Detect communities on preprocessed data
4. **Result mapping:** Convert back to original labels and dates

**Key takeaways:**
- Keep bidirectional mappings (`label_to_id` ↔ `id_to_label`)
- Sort by timestamp before normalization
- Use helper functions for reproducibility
- Visualizations can use original labels via the `labels` parameter

**Next steps:**
- Adapt the preprocessing functions to your data format
- Handle weighted interactions (add `weight` column)
- Deal with missing timestamps or duplicates
- Export results with original labels to CSV/JSON